# Laboratorio 04 — Pipeline Completa Bronze → Silver → Gold con Notificaciones

**Semana:** 05 | **Actividad de referencia:** Actividad 04  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Integra todo lo aprendido en la semana para construir una pipeline Medallion completa (Bronze → Silver → Gold) con tu dataset propio, distribuida en múltiples notebooks (uno por capa), con expectativas de calidad, parámetros, y notificaciones por webhook al completar o fallar.

> Este notebook es el **notebook Gold** y el orquestador del diseño. Los notebooks Bronze y Silver deben crearse por separado.

## Parte 1 — Descripción del dataset y arquitectura de la pipeline

1. **Nombre, fuente y URL** del dataset.
2. **Arquitectura de la pipeline:** ¿Cuántos notebooks usará la pipeline? ¿Qué genera cada uno?
3. **KPIs Gold:** ¿Qué métricas de negocio calculará la capa Gold?
4. **Webhook de notificación:** ¿A qué sistema notificarás? (Slack, Teams, email via Zapier, etc.)
5. **Preguntas de negocio finales:** Al menos 3 que respondan los datos de la capa Gold.

**Arquitectura de tu pipeline:**

```
Notebook 01 — Bronze:
  @dp.table bronze_mi_dataset — Auto Loader desde /Volumes/.../landing/

Notebook 02 — Silver:
  @dp.table silver_mi_dataset — Limpieza + expectativas
  @dp.table quarantine_mi_dataset — Registros rechazados

Notebook 03 — Gold (este): 
  @dp.materialized_view gold_kpi_1 — Primer KPI de negocio
  @dp.materialized_view gold_kpi_2 — Segundo KPI de negocio
  @dp.table gold_final — Tabla Gold consolidada
```

**Escribe aquí tu arquitectura real:**

## Parte 2 — Importaciones y parámetros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# Parámetros de pipeline — all configurable from Lakeflow UI
ENTORNO           = spark.conf.get("entorno",           "dev")
RUTA_LANDING      = spark.conf.get("ruta_landing",      "/Volumes/workspace/default/week_5/landing/")
SCHEMA_LOCATION   = spark.conf.get("schema_location",   "/Volumes/workspace/default/week_5/schema/")
WEBHOOK_URL       = spark.conf.get("webhook_url",        "")  # rellena en Lakeflow Settings
UMBRAL_NULOS      = float(spark.conf.get("umbral_nulos", "30"))  # % máximo nulos aceptado

print(f"entorno        = {ENTORNO}")
print(f"ruta_landing   = {RUTA_LANDING}")
print(f"webhook_url    = {'configurado' if WEBHOOK_URL else 'NO configurado'")

## Parte 3 — Perfil técnico del dataset (exploración interactiva)

In [ ]:
# Exploración antes de definir la pipeline
df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{RUTA_LANDING}/*.csv")

total = df.count()
print(f"Dataset: {total:,} filas | {len(df.columns)} columnas")
df.printSchema()
df.describe().show(truncate=False)

In [ ]:
# Nulos por columna
nulo_exprs = [
    F.round(
        F.sum(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), 1).otherwise(0))
        * 100.0 / total, 1
    ).alias(f"{c}")
    for c in df.columns
]
print("Porcentaje de nulos por columna:")
df.select(nulo_exprs).show(truncate=False)

In [ ]:
# Distribución de la columna de partición Gold
df.groupBy("columna_particion_gold").count().orderBy(F.col("count").desc()).show(20)

**Decisiones de diseño basadas en el perfil:**  
1. ¿Qué columnas necesitan expectativas de calidad?
2. ¿Qué columna usarás como partición en Gold?
3. ¿Qué KPIs tienen sentido con la distribución que ves?

## Parte 4 — Definición de la pipeline completa (Gold notebook)

Este notebook solo define la capa Gold. Los notebooks Bronze y Silver deben estar en `/semana_05/laboratorios/` y listarse en la configuración de la pipeline Lakeflow.

In [ ]:
# ─────────────────────────────────────────────────────────
# Gold KPI 1: resumen de negocio principal
# ─────────────────────────────────────────────────────────
@dp.materialized_view(
    comment="Gold KPI 1: resumen agregado por categoría principal."
)
def gold_kpi_resumen():
    return (
        dp.read("silver_mi_dataset_calidad")  # nombre de la tabla Silver de lab_02
        .groupBy("columna_particion_gold")
        .agg(
            F.count("*").alias("total_registros"),
            F.avg("columna_numerica").alias("promedio"),
            F.sum("columna_numerica").alias("suma_total"),
            F.max("columna_numerica").alias("maximo"),
            F.min("columna_numerica").alias("minimo")
        )
        .withColumn("_gold_ts", F.current_timestamp())
        .withColumn("_entorno", F.lit(ENTORNO))
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# Gold KPI 2: ranking dentro de cada grupo
# ─────────────────────────────────────────────────────────
from pyspark.sql.window import Window

@dp.materialized_view(
    comment="Gold KPI 2: top 10 registros por categoría."
)
def gold_kpi_top10_por_categoria():
    w = Window.partitionBy("columna_particion_gold").orderBy(F.col("columna_numerica").desc())
    return (
        dp.read("silver_mi_dataset_calidad")
        .withColumn("ranking", F.rank().over(w))
        .filter(F.col("ranking") <= 10)
        .orderBy("columna_particion_gold", "ranking")
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# Gold Final: tabla consolidada con todos los KPIs
# ─────────────────────────────────────────────────────────
@dp.table(
    comment="Gold final: tabla principal para consumo por BI/dashboards."
)
def gold_final_mi_dataset():
    df_resumen = dp.read("gold_kpi_resumen")
    # Añade más joins con otras vistas Gold si las tienes
    return df_resumen

## Parte 5 — Configuración de la pipeline multi-notebook y webhooks (teórico)

Describe aquí la configuración JSON de la pipeline completa:

```json
{
  "name": "lab05-04-pipeline-completa-mi-dataset",
  "target": "workspace.default",
  "clusters": [{"num_workers": 2}],
  "libraries": [
    {"notebook": {"path": "/semana_05/laboratorios/lab_02_autoloader_quality"}},
    {"notebook": {"path": "/semana_05/laboratorios/lab_04_pipeline_completa"}}
  ],
  "configuration": {
    "entorno":         "dev",
    "ruta_landing":    "/Volumes/workspace/default/week_5/landing/",
    "schema_location": "/Volumes/workspace/default/week_5/schema/",
    "webhook_url":     "https://hooks.slack.com/services/...",
    "umbral_nulos":    "30"
  },
  "notifications": [
    {
      "email_recipients": ["tu-email@dominio.com"],
      "alerts": ["on-update-failure", "on-flow-failure"]
    }
  ],
  "mode": "TRIGGERED"
}
```

**Edita el JSON arriba con tus valores reales. Luego responde:**
1. ¿Por qué conviene listar los notebooks en orden Bronze → Silver → Gold?
2. ¿Lakeflow infiere las dependencias entre notebooks automáticamente? ¿Cómo?
3. ¿Qué diferencia hay entre `on-update-failure` y `on-flow-failure`?

## Parte 6 — Notificación vía webhook (simulación)

In [ ]:
import requests
import json

def notificar_webhook(webhook_url: str, mensaje: str, estado: str = "success") -> None:
    """Envía una notificación al webhook configurado. Solo ejecuta si hay URL."""
    if not webhook_url:
        print(f"[SIM] Webhook no configurado — mensaje: {mensaje}")
        return
    payload = {
        "text": f":{'white_check_mark' if estado == 'success' else 'x'}: *Pipeline Lab05-04* — {mensaje}"
    }
    try:
        resp = requests.post(webhook_url, json=payload, timeout=5)
        print(f"Webhook: {resp.status_code}")
    except Exception as e:
        print(f"Error en webhook: {e}")

# Simular notificación de éxito
notificar_webhook(WEBHOOK_URL, "Pipeline completa finalizada correctamente.", "success")

## Parte 7 — Verificar resultados (notebook interactivo posterior)

In [ ]:
# Ejecutar DESPUÉS de correr la pipeline completa en Lakeflow
tablas_pipeline = [
    "workspace.default.bronze_autoloader_mi_dataset",
    "workspace.default.silver_mi_dataset_calidad",
    "workspace.default.quarantine_mi_dataset",
    "workspace.default.gold_kpi_resumen",
    "workspace.default.gold_kpi_top10_por_categoria",
    "workspace.default.gold_final_mi_dataset",
]

print(f"{'Tabla':<50} {'Filas':>10}")
print("-" * 62)
for t in tablas_pipeline:
    try:
        cnt = spark.table(t).count()
        print(f"{t.split('.')[-1]:<50} {cnt:>10,}")
    except Exception as e:
        print(f"{t.split('.')[-1]:<50} ERROR")

In [ ]:
# Mostrar Gold KPI final
spark.table("workspace.default.gold_final_mi_dataset").orderBy(F.col("total_registros").desc()).show(15, truncate=False)

## Parte 8 — Preguntas de negocio sobre la capa Gold

Responde las 3 preguntas planteadas en la Parte 1 usando la tabla Gold.

In [ ]:
# Pregunta 1:
spark.sql("""
    SELECT * FROM workspace.default.gold_final_mi_dataset
    -- añade tu condición
    LIMIT 10
""").show(truncate=False)

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:
spark.sql("""
    SELECT * FROM workspace.default.gold_kpi_top10_por_categoria
    WHERE ranking = 1
""").show(truncate=False)

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3: libre — la más útil para un stakeholder
spark.sql("""

""").show(truncate=False)

**Conclusión pregunta 3:**

## Parte 9 — Reflexión final de la semana

1. ¿Qué diferencia fundamental hay entre un Job de Databricks (semana 04) y una Pipeline Lakeflow (semana 05)?
2. ¿Qué ventajas ofrece la pipeline declarativa sobre un script imperativo para reproducibilidad y mantenimiento?
3. ¿Cuándo elegiría el patrón CDC + SCD2 sobre una pipeline batch normal con `overwrite`?
4. ¿Qué harías diferente ahora que tienes experiencia con las 5 semanas si tuvieras que diseñar este pipeline desde cero?

---

## Entrega en Git

```bash
git add semana_05/laboratorios/lab_04_pipeline_completa.ipynb
git commit -m "lab: semana05 lab04 pipeline completa B→S→G webhook Gold KPIs <nombre-dataset> - <tu-nombre>"
git push origin feature/semana05-lakeflow-<tu-nombre>
```